# 9B · Memory & Stationarity — Does This Series Remember?
### Financial Analytics — Module 9 · Lab 1

Two questions decide *whether a series can be modelled at all*, and this notebook answers both by hand:

1. **Memory (autocorrelation):** does knowing today tell you anything about tomorrow?
2. **Stationarity:** do the series' basic properties (level, spread) stay put — or wander?

> 🛡️ **Bias check:** NIFTY carries the registry's regime shift and two missing days (noted; immaterial at this frequency). Every rolling statistic below uses only past-and-present data — no centred windows in anything we'd act on.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

BASE = "data/"
px = pd.read_csv(BASE + "nifty50_prices.csv", parse_dates=["date"]).set_index("date").sort_index()
close = px["close"]
rets  = close.pct_change().dropna()

---
## 1. Memory, seen raw: the lag scatter

Plot each day against the day before. If the cloud has shape, the series remembers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(close.shift(1), close, s=4, alpha=0.4, color="#2563EB")
axes[0].set_title("PRICES: today vs yesterday - a razor line.\nPrices remember almost everything", fontsize=10)
axes[0].set_xlabel("yesterday"); axes[0].set_ylabel("today")

axes[1].scatter(rets.shift(1), rets, s=4, alpha=0.4, color="#DC2626")
axes[1].set_title("RETURNS: today vs yesterday - a shapeless cloud.\nReturns remember almost nothing", fontsize=10)
axes[1].set_xlabel("yesterday's return"); axes[1].set_ylabel("today's return")
plt.tight_layout(); plt.show()

One picture, the deepest fact in market statistics: **prices have near-perfect memory of their level; returns have almost none of their direction.** (Module 6's "prices eat their own forecasts", now visible as geometry.)

## 2. Memory, measured: the autocorrelation function (ACF)

Autocorrelation at lag k = the correlation of the series with itself, shifted k steps. Compute it by hand — one line per lag:

In [ ]:
lags = range(1, 25)
acf_price = [close.autocorr(k) for k in lags]
acf_ret   = [rets.autocorr(k) for k in lags]

# Significance band: for pure noise, ~95% of autocorrs land within +/- 2/sqrt(N)
band = 2/np.sqrt(len(rets))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for ax, vals, title, clr in [(axes[0], acf_price, "Prices: memory for months", "#2563EB"),
                             (axes[1], acf_ret, "Returns: inside the noise band", "#DC2626")]:
    ax.bar(lags, vals, color=clr, width=0.6)
    ax.axhspan(-band, band, color="grey", alpha=0.18)
    ax.set_title(title, fontsize=10); ax.set_xlabel("lag (days)")
plt.tight_layout(); plt.show()
print(f"Noise band: +/-{band:.3f}. Return autocorrs at lags 1-5: {[round(v,3) for v in acf_ret[:5]]}")

The grey band is the honesty device: for pure noise, ~95% of bars should fall inside it — so one bar poking out proves nothing (with 24 lags, chance alone pokes out about one). Prices: every bar towers. Returns: bars live in the band.

**Why practitioners still care about return ACF:** the *direction* of returns is memoryless, but their *size* is not — big days cluster with big days. Check it:

In [ ]:
acf_absret = [rets.abs().autocorr(k) for k in lags]
print("ACF of |returns| at lags 1-10:", [round(v,3) for v in acf_absret[:10]])
print()
print("All positive, all outside the band: VOLATILITY has memory even when direction doesn't.")
print("Calm clusters with calm, storm with storm - this is why Module 3's rolling vol was informative,")
print("and it is the entire foundation of volatility forecasting on real desks.")

---
## 3. Stationarity: does the series stay put?

A series is (loosely) **stationary** if its level and spread don't wander with time. Why care? Because every statistic you compute — a mean, a correlation, a regression — silently assumes the thing being measured *holds still while you measure it*. The eyeball test: rolling mean and rolling std.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 5.5), sharex="col")
for j, (s, name) in enumerate([(close, "PRICES"), (rets, "RETURNS")]):
    axes[0, j].plot(s.rolling(120).mean(), color="#2563EB")
    axes[0, j].set_title(f"{name}: 120-day rolling MEAN", fontsize=9)
    axes[1, j].plot(s.rolling(120).std(), color="#7C3AED")
    axes[1, j].set_title(f"{name}: 120-day rolling STD", fontsize=9)
plt.tight_layout(); plt.show()

Read the four panels: the price series' rolling mean **wanders relentlessly** (non-stationary in level — no surprise, it trends). Returns' rolling mean hugs zero (stationary-ish in level) — but the rolling std **steps up mid-series**: the regime change from the registry, caught red-handed by a four-line eyeball test. Returns are stationary in direction, *not* in risk.

**The fix for a wandering level is differencing** — and you've been doing it all course: `pct_change()` IS the differencing that turns non-stationary prices into workable returns. That is the real reason finance computes returns at all: not convention — *statistical necessity*.

*(Formal tests exist — ADF, KPSS — and statsmodels runs them in one line. They automate exactly the judgment you just made visually; learn the eyeball first so the test result means something.)*

---
## 4. The MA crossover — a regime describer (not yet a strategy)

In [ ]:
ma_fast, ma_slow = close.rolling(50).mean(), close.rolling(200).mean()
regime = (ma_fast > ma_slow)

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(close, color="#94A3B8", lw=0.7)
ax.plot(ma_fast, color="#EA580C", lw=1.1, label="50-day MA")
ax.plot(ma_slow, color="#0D9488", lw=1.1, label="200-day MA")
ax.fill_between(close.index, close.min(), close.max(), where=regime, color="#16A34A", alpha=0.06)
ax.set_title("Green shading = fast MA above slow ('uptrend regime'). A DESCRIPTION - trading it is Module 11's question",
             loc="left", fontweight="bold", fontsize=10)
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

flips = regime.astype(int).diff().abs().sum()
print(f"Regime flips in the sample: {int(flips)} - each flip is a potential trade, and each trade a potential cost.")

The crossover is genuinely useful **as a descriptive regime label** — "which weather are we in?" Whether the label is *tradeable* — after costs, slippage, and the whipsaw of those flips — is a completely different question, and answering it honestly is the entire agenda of Module 11. Resist the leap today; you'll have every tool to make it properly there.

### ✏️ Exercises
1. **Sales have memory:** compute the ACF of 9A's monthly sales series at lags 1–24. Where do the tall bars sit, and why exactly there? (Hint: what repeats every 12?)
2. **Difference the sales:** compute month-over-month % change of the sales series and re-run its rolling mean/std. Stationary now? What structure *survives* differencing — and is that a bug or the signal?
3. **The band test, honestly:** generate pure noise (`rng.normal(0,1,1200)`) and compute its ACF at 24 lags. How many bars escape the +/-2/sqrt(N) band by pure chance? Run it five times with different seeds. Calibrate your eye for what "nothing" looks like — the single best vaccine against seeing patterns in noise.

---
*AI disclosure: ______*

In [ ]:
# workspace
